# Lesson 02 — Exercise Solutions

Worked solutions with a short rationale after each. Run setup.sql first.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

Find all employees whose job title contains the word 'Manager'.
Expected columns: BusinessEntityID, JobTitle
Table: HumanResources.Employee

In [ ]:
SELECT BusinessEntityID, JobTitle
FROM HumanResources.Employee
WHERE JobTitle LIKE '%Manager%'
ORDER BY JobTitle;

**Why:** LIKE with % on both sides matches anywhere in the string.

## Exercise 2

List products in subcategory 1, 2, or 3 with a ListPrice above $100.
Expected columns: Name, ProductSubcategoryID, ListPrice

In [ ]:
SELECT Name, ProductSubcategoryID, ListPrice
FROM Production.Product
WHERE ProductSubcategoryID IN (1, 2, 3)
  AND ListPrice > 100
ORDER BY ListPrice DESC;

**Why:** IN for the subcategory set, AND for the price filter.

## Exercise 3

Find orders placed between 2013-01-01 and 2013-03-31 (inclusive).
Expected columns: SalesOrderID, OrderDate, TotalDue
Table: Sales.SalesOrderHeader

In [ ]:
SELECT SalesOrderID, OrderDate, TotalDue
FROM Sales.SalesOrderHeader
WHERE OrderDate BETWEEN '2013-01-01' AND '2013-03-31'
ORDER BY OrderDate;

**Why:** BETWEEN is inclusive; use DATE literals for clarity.

## Exercise 4

Find products that have NO size specified (Size IS NULL) AND no weight (Weight IS NULL).
Expected columns: ProductID, Name

In [ ]:
SELECT ProductID, Name
FROM Production.Product
WHERE Size IS NULL
  AND Weight IS NULL;

**Why:** two IS NULL conditions joined by AND.

## Exercise 5

Using OFFSET/FETCH, return rows 21–30 of products ordered by Name ascending.
Expected columns: ProductID, Name

In [ ]:
SELECT ProductID, Name
FROM Production.Product
ORDER BY Name
OFFSET 20 ROWS
FETCH NEXT 10 ROWS ONLY;

**Why:** OFFSET n ROWS skips n rows; FETCH NEXT m ROWS ONLY takes m.

## Challenges

Harder problems. They may combine this lesson's topic with
earlier lessons, and usually need 2+ concepts in one query.

### Challenge 1

Build page 3 of a product listing: 10 rows per page, ordered by
ListPrice DESC. The ordering must be fully deterministic even when prices tie.
Expected columns: ProductID, Name, ListPrice
Hint: Add a unique tie-breaker to ORDER BY; OFFSET 20 ROWS FETCH NEXT 10.

In [ ]:
SELECT ProductID, Name, ListPrice
FROM Production.Product
WHERE ListPrice > 0
ORDER BY ListPrice DESC, ProductID
OFFSET 20 ROWS FETCH NEXT 10 ROWS ONLY;

**Why:** OFFSET/FETCH handles the paging math directly — 10/page means OFFSET 20 for page 3. Determinism needs a second ORDER BY key because 56 ListPrice values have ties (lesson concept 3); ProductID is unique, so appending it breaks every tie without changing the intended price ordering. Tempting-but-wrong: ORDER BY ListPrice DESC alone — it "works" today, but a re-plan or index change can reorder tied rows between pages with no error, silently shifting which products land on which page.

### Challenge 2

Return all products that are NOT red — where "not red" must include
products whose Color is unknown (NULL). Then explain (comment) why the naive
`<> 'Red'` version returns fewer rows.
Expected columns: ProductID, Name, Color

In [ ]:
SELECT ProductID, Name, Color
FROM Production.Product
WHERE Color <> 'Red' OR Color IS NULL
ORDER BY ProductID;
-- Naive `WHERE Color <> 'Red'` alone returns 218 rows, not 466, for exactly this reason.

**Why:** "Not red" must restore the NULL-color rows a naive `<> 'Red'` drops, so OR in `Color IS NULL` explicitly. Tempting-but-wrong: `Color <> 'Red'` alone — comparing NULL to 'Red' evaluates UNKNOWN, and WHERE discards UNKNOWN exactly like FALSE, silently losing the 248 NULL-color rows (466 correct vs. 218 naive). NOT LIKE/NOT IN carry the same trap for the same reason: any predicate against NULL resolves to UNKNOWN, not FALSE, and WHERE can't tell "unknown" from "no."

### Challenge 3

Find products whose Name contains a literal percent sign or underscore
anywhere, using an ESCAPE clause. Seed check: if none exist in AdventureWorks, search
for names containing a hyphen followed by a digit instead (e.g. 'Mountain-100').
Expected columns: ProductID, Name
Hint: LIKE '%!%%' ESCAPE '!'

In [ ]:
SELECT ProductID, Name
FROM Production.Product
WHERE Name LIKE '%!%%' ESCAPE '!' OR Name LIKE '%!_%' ESCAPE '!';  -- 0 rows in this seed
SELECT ProductID, Name
FROM Production.Product
WHERE Name LIKE '%-[0-9]%'
ORDER BY ProductID;

**Why:** ESCAPE redefines a chosen character (here `!`) so the wildcard right after it is literal, not special — `LIKE '%!%%' ESCAPE '!'` looks for a literal `%`. Verified separately that this dataset has zero name matches for `%` or `_`, so the fallback (`LIKE '%-[0-9]%'`) is what actually returns rows here. Tempting-but-wrong: `LIKE '%%%'` with no ESCAPE — `%` still means "any string" without it; ESCAPE is what makes the character literal, not the choice of `!` itself.

### Challenge 4

List the orders placed in the last 30 days of available data — "last 30
days" computed from MAX(OrderDate), not hardcoded — using a half-open range
(>= start AND < end), never BETWEEN.
Expected columns: SalesOrderID, OrderDate, TotalDue
Hint: Scalar subquery or variable for the max date.

In [ ]:
DECLARE @MaxOrderDate DATETIME = (SELECT MAX(OrderDate) FROM Sales.SalesOrderHeader);
DECLARE @StartDate DATETIME = DATEADD(DAY, -29, @MaxOrderDate);
DECLARE @EndDate DATETIME = DATEADD(DAY, 1, @MaxOrderDate);

SELECT SalesOrderID, OrderDate, TotalDue
FROM Sales.SalesOrderHeader
WHERE OrderDate >= @StartDate AND OrderDate < @EndDate
ORDER BY OrderDate, SalesOrderID;

**Why:** "Last 30 days of available data" must anchor to MAX(OrderDate), not today's date — a variable holds that scalar subquery result once. A 30-day window ending on the max day is [MAX−29, MAX+1) half-open: subtracting 30 would span 31 calendar days. The `< end` upper bound sidesteps BETWEEN's inclusive-midnight trap from this lesson's Pitfalls. Tempting-but-wrong: `BETWEEN DATEADD(DAY,-30,@Max) AND @Max` — 31 days wide AND silently drops any order timestamped after midnight on the very last day.